# conv-kernel-shape — worked example 1: Read the three axes of an nn.Conv1d weight

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-kernel-shape`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A 1D convolution weight tensor follows the layout **`(out_channels, in_channels, kernel_width)`** — OC first, IC second, then the single spatial axis KW. Just like Conv2d but with one fewer kernel dimension. The total number of weight scalars (excluding bias) is `OC * IC * KW`.

## Worked solution

**Step 1 — build a concrete module.** We make `nn.Conv1d(in_channels=4, out_channels=10, kernel_size=7)`. The constructor takes `(in, out, kernel)` order, but the *stored* `weight` tensor is OC-first.

**Step 2 — inspect the shape.** `conv.weight.shape` is `(10, 4, 7)`. The axes are, in order: `OC=10`, `IC=4`, `KW=7`. Reading them positionally (not from `conv.out_channels`) forces us to internalize the layout rather than trust the module's bookkeeping.

**Step 3 — unpack with tuple assignment.** `OC, IC, KW = conv.weight.shape` cleanly names all three. This works precisely because there are exactly three axes for a 1D conv.

**Step 4 — param count.** Weights only (no bias) is the product `OC * IC * KW = 10 * 4 * 7 = 280`. We cast to plain `int` so the returned dict holds Python ints, not 0-dim tensors or `torch.Size` entries.

In [ ]:
def conv1d_weight_facts(conv) -> dict:
    OC, IC, KW = conv.weight.shape
    return {
        'out_channels':    int(OC),
        'in_channels':     int(IC),
        'kernel_width':    int(KW),
        'n_weight_params': int(OC * IC * KW),
    }

t.manual_seed(0)
conv = t.nn.Conv1d(in_channels=4, out_channels=10, kernel_size=7)
facts = conv1d_weight_facts(conv)
print(facts)